# Gravit Grover: Proof of Concept
## Demonstration of Gower + Multiplicative Weights + Amplification

In this notebook, we demonstrate the basic mechanics of the Gravit Grover Engine:
1. **Gower Similarity**: evaluating closeness of hypotheses
2. **Multiplicative Weights**: probabilistic updating
3. **Amplification**: enhancing strong hypotheses
4. **Convergence**: convergence to the optimal hypothesis

In [ ]:
# Imports
import sys
sys.path.append('../../')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List

# Our modules
from core.gower.gower_distance import GowerDistance, GowerConfig, ConsistencyScorer
from core.probability.multiplicative_weights import MultiplicativeWeights, MWUConfig

# Visualization setup
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['font.size'] = 12

## 1. Creating the Hypothesis Space

In [ ]:
# Define hypotheses
hypotheses_text = [
    'Raise taxes to fight inflation',
    'Lower taxes to stimulate the economy',
    'Print money to cover debt',
    'Reduce government spending',
    'Introduce price controls',
    'Increase interest rates',
    'Stimulate exports',
    'Devalue the national currency'
]

# Create features for hypotheses (in practice, these would be embeddings or expert scores)
# Features: [inflation, unemployment, GDP growth, inequality, government debt]
hypotheses_features = np.array([
    [0.8, 0.3, 0.1, 0.2, 0.4],  # Raise taxes
    [0.2, 0.1, 0.9, 0.1, 0.6],  # Lower taxes
    [0.9, 0.7, 0.2, 0.8, 0.1],  # Print money
    [0.1, 0.5, 0.3, 0.1, 0.2],  # Reduce spending
    [0.7, 0.6, 0.2, 0.5, 0.3],  # Price controls
    [0.6, 0.4, 0.4, 0.3, 0.5],  # Increase rates
    [0.3, 0.2, 0.7, 0.2, 0.7],  # Stimulate exports
    [0.5, 0.6, 0.3, 0.7, 0.2],  # Devaluation
])

# Feature types
feature_types = ['numeric', 'numeric', 'numeric', 'numeric', 'numeric']

print(f"Total hypotheses: {len(hypotheses_text)}")
print(f"Feature dimensionality: {hypotheses_features.shape[1]}")

## 2. Gower Similarity Layer

Computing the similarity matrix between hypotheses.

In [ ]:
# Initialize Gower
gower_config = GowerConfig(normalize_numeric=True)
gower = GowerDistance(gower_config)
gower.fit(hypotheses_features, feature_types)

# Compute similarity matrix
similarity_matrix = gower.compute_similarity_matrix(hypotheses_features)

# Visualize similarity matrix
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Similarity matrix
im = ax[0].imshow(similarity_matrix, cmap='Blues', vmin=0, vmax=1)
ax[0].set_title('Gower Similarity Matrix')
ax[0].set_xlabel('Hypotheses')
ax[0].set_ylabel('Hypotheses')
plt.colorbar(im, ax=ax[0])

# Similarity distribution
similarities_flat = similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)]
ax[1].hist(similarities_flat, bins=20, edgecolor='black', alpha=0.7)
ax[1].set_title('Distribution of Pairwise Similarities')
ax[1].set_xlabel('Similarity')
ax[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Print statistics
print(f"Average similarity: {np.mean(similarities_flat):.3f}")
print(f"Minimum similarity: {np.min(similarities_flat):.3f}")
print(f"Maximum similarity: {np.max(similarities_flat):.3f}")

## 3. Consistency Scoring

Computing the consistency of each hypothesis relative to others.

In [ ]:
# Create Consistency Scorer
consistency_scorer = ConsistencyScorer(gower, threshold=0.5)

# Compute consistency
consistent_subset, consistency_scores = consistency_scorer.get_consistent_subset(hypotheses_features)

# Visualize
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Consistency per hypothesis
ax[0].bar(range(len(hypotheses_text)), consistency_scores, color='teal', alpha=0.7)
ax[0].set_title('Consistency Score per Hypothesis')
ax[0].set_xlabel('Hypothesis')
ax[0].set_ylabel('Consistency')
ax[0].set_xticks(range(len(hypotheses_text)))
ax[0].set_xticklabels(hypotheses_text, rotation=45, ha='right')
ax[0].axhline(y=0.5, color='red', linestyle='--', label='Threshold')
ax[0].legend()

# Hypotheses ranked by consistency
sorted_idx = np.argsort(consistency_scores)[::-1]
sorted_text = [hypotheses_text[i] for i in sorted_idx]
sorted_scores = [consistency_scores[i] for i in sorted_idx]

ax[1].barh(range(len(sorted_text)), sorted_scores, color='coral', alpha=0.7)
ax[1].set_title('Hypotheses Ranked by Consistency')
ax[1].set_xlabel('Consistency')
ax[1].set_ylabel('Hypothesis')
ax[1].set_yticks(range(len(sorted_text)))
ax[1].set_yticklabels(sorted_text)
ax[1].axvline(x=0.5, color='red', linestyle='--', label='Threshold')
ax[1].legend()

plt.tight_layout()
plt.show()

# Display top hypotheses
print("Top-3 consistent hypotheses:")
for i in range(min(3, len(sorted_text))):
    print(f"{i+1}. {sorted_text[i]} (score: {sorted_scores[i]:.3f})")

## 4. Multiplicative Weights

Simulating iterative probability updates.

In [ ]:
# Initialize MWU
mwu_config = MWUConfig(
    learning_rate=2.0,
    temperature=1.0,
    use_entropy_regularization=True,
    entropy_coefficient=0.1
)
mwu = MultiplicativeWeights(mwu_config)

# Initial uniform distribution
n_hypotheses = len(hypotheses_text)
probs = np.ones(n_hypotheses) / n_hypotheses

# Simulation iterations
n_iterations = 10
history_probs = [probs.copy()]
history_entropy = [mwu.compute_entropy(probs)]

for iteration in range(n_iterations):
    # Generate scores (in practice, these come from validators)
    # Use consistency as the score
    scores = np.array(consistency_scores) * (1 + 0.1 * np.random.randn(n_hypotheses))
    scores = np.maximum(scores, 0)
    
    # Update probabilities
    probs, entropy = mwu.update(probs, scores, return_entropy=True)
    
    # Save history
    history_probs.append(probs.copy())
    history_entropy.append(entropy)

# Visualize distribution evolution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Probability evolution
for i in range(n_hypotheses):
    probs_history = [p[i] for p in history_probs]
    axes[0, 0].plot(range(len(probs_history)), probs_history, 
                    label=hypotheses_text[i][:15], alpha=0.8)
axes[0, 0].set_title('Evolution of Hypothesis Probabilities')
axes[0, 0].set_xlabel('Iteration')
axes[0, 0].set_ylabel('Probability')
axes[0, 0].legend(loc='center left', bbox_to_anchor=(1, 0.5))

# 2. Entropy
axes[0, 1].plot(history_entropy, color='red', linewidth=2)
axes[0, 1].set_title('Entropy over Time')
axes[0, 1].set_xlabel('Iteration')
axes[0, 1].set_ylabel('Entropy')
axes[0, 1].grid(True)

# 3. Final distribution
final_probs = history_probs[-1]
indices = np.argsort(final_probs)[::-1][:5]
top_texts = [hypotheses_text[i] for i in indices]
top_probs = [final_probs[i] for i in indices]

axes[1, 0].barh(range(len(top_texts)), top_probs, color='green', alpha=0.7)
axes[1, 0].set_title('Top-5 Hypotheses (Final Distribution)')
axes[1, 0].set_xlabel('Probability')
axes[1, 0].set_ylabel('Hypothesis')
axes[1, 0].set_yticks(range(len(top_texts)))
axes[1, 0].set_yticklabels(top_texts)

# 4. KL divergence from uniform
uniform = np.ones(n_hypotheses) / n_hypotheses
kl_divergences = [mwu.compute_kl_divergence(p, uniform) for p in history_probs]

axes[1, 1].plot(kl_divergences, color='purple', linewidth=2)
axes[1, 1].set_title('KL Divergence from Uniform')
axes[1, 1].set_xlabel('Iteration')
axes[1, 1].set_ylabel('KL Divergence')
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

# Display results
print("\n=== FINAL RESULTS ===")
print(f"Iterations: {n_iterations}")
print(f"Initial entropy: {history_entropy[0]:.3f}")
print(f"Final entropy: {history_entropy[-1]:.3f}")
print(f"Entropy reduction: {history_entropy[0] - history_entropy[-1]:.3f}")

print("\nTop-3 hypotheses:")
for i, (text, prob) in enumerate(zip(top_texts, top_probs)):
    print(f"{i+1}. {text}")
    print(f"   Probability: {prob:.3f}")

## 5. Grover Amplification

Demonstrating the amplification effect.

In [ ]:
def grover_amplification(probabilities: np.ndarray, 
                         scores: np.ndarray,
                         gamma: float = 2.0) -> np.ndarray:
    """
    Grover amplification: exponential increase in weight of best hypotheses.
    """
    # Normalize scores
    normalized_scores = scores / np.max(scores + 1e-10)
    
    # Amplification
    amplified = probabilities * (normalized_scores ** gamma)
    
    # Normalize
    return amplified / np.sum(amplified)

# Apply Grover Amplification to the final distribution
final_scores = np.array(consistency_scores)
amplified_probs = grover_amplification(final_probs, final_scores, gamma=3.0)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Compare distributions
x = np.arange(n_hypotheses)
width = 0.35

axes[0].bar(x - width/2, final_probs, width, label='Before Amplification', alpha=0.8)
axes[0].bar(x + width/2, amplified_probs, width, label='After Amplification', alpha=0.8)
axes[0].set_title('Grover Amplification Effect')
axes[0].set_xlabel('Hypothesis Index')
axes[0].set_ylabel('Probability')
axes[0].legend()

# Probability ratio
ratio = amplified_probs / (final_probs + 1e-10)
axes[1].bar(x, ratio, color='orange', alpha=0.7)
axes[1].set_title('Amplification Ratio')
axes[1].set_xlabel('Hypothesis Index')
axes[1].set_ylabel('Ratio (Amplified / Original)')
axes[1].axhline(y=1, color='red', linestyle='--')

plt.tight_layout()
plt.show()

# Display results
best_original = np.argmax(final_probs)
best_amplified = np.argmax(amplified_probs)

print(f"Best hypothesis before amplification: {hypotheses_text[best_original]}")
print(f"Best hypothesis after amplification: {hypotheses_text[best_amplified]}")
print(f"Probability before: {final_probs[best_original]:.3f}")
print(f"Probability after: {amplified_probs[best_amplified]:.3f}")

## 6. Conclusions

**What we demonstrated:**

1. **Gower Similarity** allows measuring closeness of hypotheses in mixed feature spaces
2. **Consistency Scoring** identifies consistent hypotheses, filtering out anomalies
3. **Multiplicative Weights** evolves the probability distribution, increasing weight of good hypotheses
4. **Grover Amplification** further enhances dominant hypotheses
5. **Convergence** is achieved through entropy reduction and KL divergence increase

**Next steps:**

1. Integration with real LLMs for hypothesis generation
2. Implementation of Gossip Protocol for distributed synchronization
3. Scaling to thousands of nodes
4. Testing on real-world tasks (QA, Medical Diagnosis, Policy Design)